<a href="https://colab.research.google.com/github/dyjdlopez/intro_2_quantum/blob/main/cuda-q/pythonasia26/cudaq_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Quantum Programming with CUDA-Q
### CUDA-Q Fundamentals — Bell State to Parameterized Circuits

**Python Asia 2026** · Dylan Josh Lopez, M.Sc

---

Welcome! By the end of this notebook, you will be able to:

- ✅ Write and run a CUDA-Q kernel using the `@cudaq.kernel` decorator
- ✅ Use `cudaq.sample()` to collect and interpret a probability histogram
- ✅ Build a Bell state and understand entanglement through code
- ✅ Scale entanglement to N qubits with the GHZ state
- ✅ Use `cudaq.observe()` to compute an expectation value
- ✅ Swap execution backends with a single line — CPU, GPU, or QPU

> **No quantum physics background needed.** If you can write a Python function, you can write a quantum kernel.

---


In this notebook we will look at initial steps in programming quantum algorithms with CUDA-Q.

CUDA-Q is **NVIDIA’s platform** for accelerating quantum computing using GPUs. It enables efficient execution of quantum algorithms by integrating classical and quantum processing. Designed for both beginners and experts, CUDA-Q supports **quantum simulations, optimization, and AI applications**, allowing users to explore quantum computing without direct access to quantum hardware.
For more details, visit the **[CUDA-Q official website](https://developer.nvidia.com/cuda-quantum)**.

![image](https://www.nvidia.com/content/nvidiaGDC/us/en_US/solutions/quantum-computing/_jcr_content/root/responsivegrid/nv_container_1856196339/nv_container/nv_image.coreimg.svg/1731940253298/cuda-quantum-diagram.svg)

 Although CUDA-Q is not the  initial choice for quantum algorithim programming (unlike Qiskit), its practical use with GPU acceleration allows it to become a good choice for near real-time simulation. Since the flow of the course focuses on the practical applications of quantum computing, it is essential to employ its deployability and interoperability with GPU hardware.

 ![image](https://developer.download.nvidia.com/images/cuda-q/quantum-computing-cuda-q-diagram-4740950-chart.svg)


## Section 0 — Setup

Let's install CUDA-Q. This takes about 60–90 seconds on Colab. We'll talk through the qubit analogy while it runs.

In [ ]:
!pip install cudaq

In [ ]:
import cudaq
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
cudaq.set_random_seed(42)

print('CUDA-Q version:', cudaq.__version__)
print('Ready!')

---
## Section 1: Qubits, Gates, Circuits, and Measurement

We'll build up piece by piece:

1. Allocating quantum memory (`cudaq.qvector`)
2. Applying gates and inspecting the state (`cudaq.get_state`)
3. Visualising the circuit (`cudaq.draw`)
4. Measuring and reading probabilities (`cudaq.sample`)

### 1.1 Quantum Memory

In classical computing, memory holds bits. In CUDA-Q, **quantum memory** holds qubits using `cudaq.qvector(n)`.

A **qubit** starts in state `|0⟩` by default — the quantum equivalent of initialising a variable to zero. Unlike a classical bit which is always exactly `0` or `1`, a qubit can exist in a **superposition** of both:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

where $|\alpha|^2 + |\beta|^2 = 1$. Think of it like a coin spinning in the air — it has not landed yet.

In CUDA-Q, all quantum operations must live inside a **kernel** — a function decorated with `@cudaq.kernel`. This tells CUDA-Q to compile and optimise the function for quantum execution.

In [ ]:
@cudaq.kernel
def quantum_memory():
    q1 = cudaq.qvector(1)  # allocate 1 qubit
    q3 = cudaq.qvector(3)  # allocate 3 qubits

# We can inspect the default state with cudaq.get_state()
# Every qubit starts in |0> — the statevector is [1, 0]
print('State of a single qubit (no gates applied):')
print(cudaq.get_state(quantum_memory))
print()
print('This is the statevector: amplitude of |0000> = 1.0, all others = 0.0')

### 1.2 Applying Gates and Inspecting the State

Gates are operations that transform qubit states. The most important single-qubit gates are:

| Gate | CUDA-Q | What it does |
|---|---|---|
| **Hadamard** | `h(q[i])` | Creates equal superposition: $|0\rangle \rightarrow \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ |
| **Pauli-X** | `x(q[i])` | Bit flip: $|0\rangle \leftrightarrow |1\rangle$ — the quantum NOT gate |
| **Pauli-Z** | `z(q[i])` | Phase flip: $|1\rangle \rightarrow -|1\rangle$, leaves $|0\rangle$ unchanged |

After applying gates, we can inspect the **statevector** — the exact quantum amplitudes — using `cudaq.get_state()`. This is only possible on a simulator (real hardware cannot read the statevector directly).

In [ ]:
# Pauli-X gate: flips |0> to |1>
@cudaq.kernel
def apply_x():
    q = cudaq.qvector(1)
    x(q[0])   # flip from |0> to |1>

print('After X gate — qubit is now in |1>:')
print(cudaq.get_state(apply_x))
print('Statevector: amplitude of |1> = 1.0')

In [ ]:
# Hadamard gate: creates equal superposition
@cudaq.kernel
def apply_h():
    q = cudaq.qvector(1)
    h(q[0])   # put qubit into superposition

print('After H gate — qubit is in superposition:')
print(cudaq.get_state(apply_h))
print()
print('Statevector: amplitude of |0> = 1/√2 ≈ 0.707, amplitude of |1> = 1/√2 ≈ 0.707')
print('Probability of measuring |0> = (0.707)^2 ≈ 0.5')
print('Probability of measuring |1> = (0.707)^2 ≈ 0.5')

In [ ]:
# We can chain gates — let's apply both X then H on one qubit
# and H on a second qubit, to see a 2-qubit statevector
@cudaq.kernel
def two_qubit_state():
    q = cudaq.qvector(2)
    x(q[0])   # flip qubit 0 to |1>
    h(q[1])   # put qubit 1 in superposition

print('2-qubit statevector after X(q0) and H(q1):')
print(cudaq.get_state(two_qubit_state))
print()
print('We expect non-zero amplitudes only at |10> and |11>')
print('(q0=1 is fixed, q1 is in superposition)')

### 1.3 Visualising the Circuit

Reading statevectors tells us the *output* but not the *structure*. `cudaq.draw()` prints an ASCII diagram of the circuit — showing exactly which gates were applied, in order, on each qubit.

This is how quantum circuits are conventionally drawn:
- Each horizontal line is a qubit (time flows left to right)
- Each box on a line is a gate applied to that qubit
- Vertical connections between lines indicate multi-qubit gates

In [ ]:
# Draw the single-qubit Hadamard circuit
print('Circuit: H gate on one qubit')
print(cudaq.draw(apply_h))

In [ ]:
# Draw the two-qubit circuit
print('Circuit: X on q0, H on q1')
print(cudaq.draw(two_qubit_state))

In [ ]:
# Let's build a slightly more interesting single-qubit circuit
# and see what the diagram looks like with chained gates
@cudaq.kernel
def gate_chain():
    q = cudaq.qvector(1)
    h(q[0])   # superposition
    z(q[0])   # phase flip
    h(q[0])   # back out of superposition — this is effectively a Z gate!

print('Circuit: H → Z → H')
print(cudaq.draw(gate_chain))
print()
print('Statevector after H→Z→H:')
print(cudaq.get_state(gate_chain))
print('(H Z H = X — the gate sequence is equivalent to a Pauli-X!)')

### 1.4 Measurement and Probabilities

The statevector shows us amplitudes, but on real quantum hardware we can only **measure** — which collapses the qubit to `|0⟩` or `|1⟩` with a probability given by the amplitude squared.

- `mz(q)` — measures all qubits in the Z basis, collapsing the state
- `cudaq.sample(kernel, shots_count=N)` — runs the circuit N times and tallies outcomes

More shots → closer approximation to the true probability distribution. A single shot gives one answer; a thousand shots gives us the full picture.

> **Why can't we just read the statevector on hardware?** Measuring destroys superposition — we only see one outcome per shot. `cudaq.get_state()` is a simulator luxury; `cudaq.sample()` works everywhere.

In [ ]:
# Add measurement to the Hadamard circuit
@cudaq.kernel
def single_qubit_measured():
    q = cudaq.qvector(1)
    h(q[0])   # superposition
    mz(q)     # measure — collapses to |0> or |1>

# Run 1000 times
result = cudaq.sample(single_qubit_measured, shots_count=1000)

print('Raw result dump:')
result.dump()
print()
print(f'Most probable outcome : {result.most_probable()}')
print(f'Pr(|0>) = {result.probability("0"):.3f}')
print(f'Pr(|1>) = {result.probability("1"):.3f}')

---
## Section 2 - Bell States

Superposition is interesting, but **entanglement** is where things get strange.

A **Bell state** is a 2-qubit state where the two qubits are perfectly correlated — measuring one instantly tells us the other, no matter how far apart they are.

We create it with two gates:
1. **H** on qubit 0 — puts it in superposition
2. **CNOT (`cx`)** — flips qubit 1 *if and only if* qubit 0 is `|1⟩`

```
q[0] : ─── H ──── ● ────
                  │
q[1] : ────────── X ────
```

The result is: $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$

We should **never** see `01` or `10`.

In [ ]:
@cudaq.kernel
def bell():
    q = cudaq.qvector(2)
    h(q[0])          # superposition on qubit 0
    cx(q[0], q[1])   # entangle: flip q[1] if q[0] is |1>
    mz(q)            # measure both

result = cudaq.sample(bell, shots_count=1000)
print('Bell state measurement (1000 shots):')
result.dump()

print()
print('Circuit diagram:')
print(cudaq.draw(bell))

### ✏️ Exercise — Flip the Bell state to `|11⟩`

The Bell state above starts from `|00⟩`. Can you modify the kernel so that the dominant outcomes are `|11⟩` and `|00⟩`, but the base entangled state shifts to `|11⟩`?

**Hint:** Think about what the `x` gate does to a qubit *before* it enters superposition.

**Expected output:** `{ 11:~500, 00:~500 }` — same distribution, but with `|11⟩` as the base.

In [ ]:
@cudaq.kernel
def bell_flipped():
    q = cudaq.qvector(2)
    # YOUR CODE HERE
    # Hint: add one gate before h(q[0])
    h(q[0])
    cx(q[0], q[1])
    mz(q)

result = cudaq.sample(bell_flipped, shots_count=1000)
result.dump()

---
## Section 3: GHZ State- Scaling Entanglement to N Qubits

The Bell state entangles 2 qubits. The **GHZ (Greenberger–Horne–Zeilinger) state** generalises this to N qubits:

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|00\ldots0\rangle + |11\ldots1\rangle)$$

The recipe is simple: H on qubit 0, then a chain of CNOT gates.

Notice that the code barely changes — we just loop. This is the beauty of CUDA-Q.

In [ ]:
@cudaq.kernel
def ghz(n_qubits: int):
    q = cudaq.qvector(n_qubits)
    h(q[0])
    for i in range(n_qubits - 1):
        cx(q[i], q[i + 1])
    mz(q)

# Run for n=3
result_3 = cudaq.sample(ghz, 3, shots_count=1000)
print('GHZ (3 qubits):')
result_3.dump()
print()
print('Circuit diagram (n=3):')
print(cudaq.draw(ghz, 3))

---
## Section 4 - `cudaq.sample()`, How Many Shots?

A quantum circuit gives a *probability distribution* over outcomes — not a single deterministic answer. `cudaq.sample()` runs the circuit many times ("shots") and returns the empirical distribution.

More shots → closer approximation to the true distribution. Let's see this in action.

In [ ]:
shot_counts = [100, 1000, 10000]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, shots in zip(axes, shot_counts):
    result = cudaq.sample(bell, shots_count=shots)
    labels = ['00', '11']
    vals = [result[b] if b in result else 0 for b in labels]
    ax.bar(labels, vals, color=['#6d4d8e', '#76b900'])
    ax.set_title(f'{shots:,} shots')
    ax.set_ylabel('Count')
    ax.set_ylim(0, shots * 0.7)

plt.suptitle('Bell state — more shots → smoother distribution', fontsize=11)
plt.tight_layout()
plt.show()

# Useful result methods
result_1k = cudaq.sample(bell, shots_count=1000)
print('Most probable bitstring:', result_1k.most_probable())
print('Probability of |00>:    ', round(result_1k.probability('00'), 3))
print('Probability of |11>:    ', round(result_1k.probability('11'), 3))

---
## Section 5 - Expectation Values

Beyond counting bitstrings, we often want to compute the **expected energy** of a quantum state under a given Hamiltonian — written $\langle\psi|H|\psi\rangle$.

`cudaq.observe()` does this in one call. It returns a single float — the expectation value.

This is the number that a **variational algorithm** (like QAOA or VQE) tries to minimise. We'll use this in Notebook 2.

Here we define a simple ZZ Hamiltonian and a parameterised 2-qubit ansatz with an `rx(theta)` rotation.

In [ ]:
from cudaq import spin

# Hamiltonian: measure the correlation between qubit 0 and qubit 1
hamiltonian = spin.z(0) * spin.z(1)

@cudaq.kernel
def ansatz(theta: float):
    q = cudaq.qvector(2)
    h(q[0])
    rx(theta, q[1])   # parameterised rotation
    cx(q[0], q[1])

# Compute the expectation value at theta = 0.5 radians
theta_val = 0.5
energy = cudaq.observe(ansatz, hamiltonian, theta_val)
print(f'<Z0 Z1> at theta={theta_val}: {energy.expectation():.4f}')
print()
print('This single float is the cost a classical optimizer will try to minimise.')

### ✏️ Exercise — Sweep theta and plot the variational landscape

The expectation value changes as `theta` changes. Let's sweep from `0` to `2π` and plot the **variational landscape** — the curve a classical optimizer would navigate.

**Expected output:** A sinusoidal curve showing how the energy oscillates with the rotation angle.

In [ ]:
# YOUR CODE HERE
# 1. Create a list of 50 theta values from 0 to 2*pi using np.linspace
# 2. For each theta, call cudaq.observe(ansatz, hamiltonian, theta).expectation()
# 3. Plot energies vs thetas using matplotlib

thetas = np.linspace(0, 2 * np.pi, 50)
energies = []

# ... your loop here ...

# plt.plot(thetas, energies)
# plt.xlabel('theta (radians)')
# plt.ylabel('<Z0 Z1> expectation value')
# plt.title('Variational landscape')
# plt.show()

In [ ]:
# SOLUTION
thetas = np.linspace(0, 2 * np.pi, 50)
energies = []

for theta in thetas:
    e = cudaq.observe(ansatz, hamiltonian, theta).expectation()
    energies.append(e)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(thetas, energies, color='#76b900', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('theta (radians)')
ax.set_ylabel('<Z0 Z1> expectation value')
ax.set_title('Variational landscape — the curve QAOA navigates')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
plt.tight_layout()
plt.show()

---
## Section 6 Backend Swap: One Line to GPU or QPU

Every kernel we've written runs on a CPU simulator. To move to a GPU or real quantum hardware, we change **one line** — the kernel code stays identical.

| Target | Code | What it runs on |
|---|---|---|
| `'qpp-cpu'` | default | CPU simulator (Colab-safe) |
| `'nvidia'` | uncomment if on GPU runtime | NVIDIA GPU via cuQuantum |
| `'ionq'` | requires API key | IonQ trapped-ion QPU |
| `'quantinuum'` | requires API key | Quantinuum H-series QPU |

This is CUDA-Q's QPU-agnostic design — 75% of public QPUs are supported.

In [ ]:
import time
# Run on CPU simulator (always works on Colab free tier)
cudaq.set_target('qpp-cpu')


t0_cpu = time.time()
result_cpu = cudaq.sample(bell, shots_count=1000)
t1_cpu = time.time()
result_cpu.dump()

t0_gpu = time.time()
cudaq.set_target('nvidia')
t1_gpu = time.time()
result_gpu = cudaq.sample(bell, shots_count=1000)

print(f'CPU-based Entanglement Simulation time: {t1_cpu-t0_cpu:.6f}')
print(f'GPU-based Entanglement Simulation time: {t1_gpu-t0_gpu:.6f}')

In [ ]:
# Reset to CPU for the rest of the notebook
cudaq.set_target('qpp-cpu')
print('\nBackend reset to qpp-cpu')

---
## Section 7 Wrap-Up

Here's what we covered in this notebook:

| API | What it does |
|---|---|
| `@cudaq.kernel` | Defines a quantum circuit as a Python function |
| `cudaq.sample()` | Runs the circuit N times, returns a probability histogram |
| `cudaq.observe()` | Computes the expectation value $\langle H \rangle$ — the VQA cost function |
| `cudaq.set_target()` | Swaps the execution backend — CPU, GPU, or QPU |

### What's next?

In **Notebook 2** we put `observe()` to work inside a real optimization loop — running **QAOA for Max Cut** using `cudaq_solvers`. The same Python, a real graph problem.

- 📓 **Notebook 2:** QAOA for Max Cut — pyqubo + cudaq_solvers + CUDA-Q
- 📦 **More examples:** [github.com/NVIDIA/cuda-q-academic](https://github.com/NVIDIA/cuda-q-academic)
- 🌏 **QCSP:** [qcsp.ph](https://qcsp.ph)

---
$$_{\text{Dylan Josh Lopez| Python Asia 2026}}$$